# Torque-Coefficient Integration

This notebook preserves the full numerical sector integrations used to compute `c11`, `c22`, `c12`, and `c21`. The obsolete comparison against a different Mathematica disk geometry has been removed. Change geometry-derived sector parameters deliberately and rerun all cells before exporting new coefficients.

In [ ]:
# ============================================================
# Three-disk geometry:
# areas, centroids, and radial edge-to-edge width
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

from shapely.geometry import Point, LineString

try:
    import pandas as pd
except ImportError:
    pd = None


# ============================================================
# 1. Create disks
# ============================================================

def make_disk(center, radius, resolution=512):
    """
    Create a disk using shapely.

    Parameters
    ----------
    center : tuple
        (x, y) center of the disk
    radius : float
        Disk radius
    resolution : int
        Number of segments per quadrant used to approximate the circle
    """
    x, y = center

    try:
        # Shapely >= 2
        return Point(x, y).buffer(radius, quad_segs=resolution)
    except TypeError:
        # Shapely < 2
        return Point(x, y).buffer(radius, resolution=resolution)


# ============================================================
# 2. Compute regions
# ============================================================

def compute_regions(disks):
    """
    Build the regions:

    Triple = D1 ∩ D2 ∩ D3
    BlueGreenMinusRed = (D2 ∩ D3) without Triple
    RedBlueMinusGreen = (D1 ∩ D2) without Triple
    """
    D1, D2, D3 = disks

    triple = D1.intersection(D2).intersection(D3)
    blue_green_minus_red = D2.intersection(D3).difference(triple)
    red_blue_minus_green = D1.intersection(D2).difference(triple)

    return {
        "Triple": triple,
        "BlueGreenMinusRed": blue_green_minus_red,
        "RedBlueMinusGreen": red_blue_minus_green,
    }


# ============================================================
# 3. Area and centroid
# ============================================================

def region_area_centroid(region, area_tol=1e-15):
    """
    Return the area and centroid of a region.
    If the region is empty, return np.nan values.
    """
    if region is None or region.is_empty or region.area <= area_tol:
        return np.nan, (np.nan, np.nan)

    centroid = region.centroid
    return region.area, (centroid.x, centroid.y)


# ============================================================
# 4. Helpers for radial intersection
# ============================================================

def _long_line_through(origin, point, geom, scale=10.0):
    """
    Create a sufficiently long line passing through origin and point.
    """
    O = np.array(origin, dtype=float)
    G = np.array(point, dtype=float)

    v = G - O
    norm = np.linalg.norm(v)

    if norm == 0:
        return None

    u = v / norm

    minx, miny, maxx, maxy = geom.bounds
    diag = np.sqrt((maxx - minx)**2 + (maxy - miny)**2)

    L = max(scale * diag, scale * norm, 1.0)

    p1 = O - L * u
    p2 = O + L * u

    return LineString([tuple(p1), tuple(p2)])


def _extract_line_segments(geom):
    """
    Extract all LineString segments from an intersection geometry.
    """
    if geom is None or geom.is_empty:
        return []

    if geom.geom_type == "LineString":
        return [geom] if geom.length > 0 else []

    if geom.geom_type == "MultiLineString":
        return [g for g in geom.geoms if g.length > 0]

    if geom.geom_type == "GeometryCollection":
        segments = []
        for g in geom.geoms:
            segments.extend(_extract_line_segments(g))
        return segments

    return []


def radial_width_through_centroid(region, origin, centroid, tol=1e-10):
    """
    Compute the edge-to-edge radial width of a region.

    Procedure:
    - Build the line through origin and centroid.
    - Intersect that line with the region.
    - If multiple segments appear, select the one containing the centroid.
    - If due to numerical tolerance the centroid is not exactly on a segment,
      choose the segment whose midpoint is closest to the centroid.
    """
    if region is None or region.is_empty:
        return np.nan, (np.nan, np.nan), (np.nan, np.nan), None

    if centroid is None or np.any(np.isnan(centroid)):
        return np.nan, (np.nan, np.nan), (np.nan, np.nan), None

    O = np.array(origin, dtype=float)
    G = np.array(centroid, dtype=float)

    if np.linalg.norm(G - O) < tol:
        return np.nan, (np.nan, np.nan), (np.nan, np.nan), None

    radial_line = _long_line_through(origin, centroid, region)

    if radial_line is None:
        return np.nan, (np.nan, np.nan), (np.nan, np.nan), None

    intersection = region.intersection(radial_line)
    segments = _extract_line_segments(intersection)

    if len(segments) == 0:
        return np.nan, (np.nan, np.nan), (np.nan, np.nan), intersection

    centroid_point = Point(G[0], G[1])

    chosen_segment = None

    # First try: segment containing the centroid
    for seg in segments:
        if seg.distance(centroid_point) <= tol:
            chosen_segment = seg
            break

    # Fallback: segment whose midpoint is closest to the centroid
    if chosen_segment is None:
        def midpoint_distance(seg):
            mid = seg.interpolate(0.5, normalized=True)
            return mid.distance(centroid_point)

        chosen_segment = min(segments, key=midpoint_distance)

    coords = list(chosen_segment.coords)
    edge_point_1 = coords[0]
    edge_point_2 = coords[-1]

    radial_width = Point(edge_point_1).distance(Point(edge_point_2))

    return radial_width, edge_point_1, edge_point_2, intersection


# ============================================================
# 5. Full analysis
# ============================================================

def analyze_three_disks(
    C1, R1,
    C2, R2,
    C3, R3,
    resolution=512,
    origin_index=1
):
    """
    Analyze three disks and return:
    - disks
    - regions
    - results table

    By default, the radial origin is C1.
    """
    D1 = make_disk(C1, R1, resolution=resolution)
    D2 = make_disk(C2, R2, resolution=resolution)
    D3 = make_disk(C3, R3, resolution=resolution)

    disks = [D1, D2, D3]
    centers = [C1, C2, C3]
    origin = centers[origin_index - 1]

    regions = compute_regions(disks)

    results = []

    for name, region in regions.items():
        area, centroid = region_area_centroid(region)

        radial_width, p_edge_1, p_edge_2, _ = radial_width_through_centroid(
            region=region,
            origin=origin,
            centroid=centroid
        )

        results.append({
            "Region": name,
            "Area": area,
            "Centroid_x": centroid[0],
            "Centroid_y": centroid[1],
            "Radial_width": radial_width,
            "Edge_point_1": p_edge_1,
            "Edge_point_2": p_edge_2,
        })

    if pd is not None:
        results_table = pd.DataFrame(results)
    else:
        results_table = results

    return disks, regions, results_table


# ============================================================
# 6. Plotting utilities
# ============================================================

def _plot_polygon(ax, geom, color, alpha=0.4, edgecolor="black", linewidth=1.0):
    """
    Plot a Polygon or MultiPolygon.
    """
    if geom is None or geom.is_empty:
        return

    if geom.geom_type == "Polygon":
        x, y = geom.exterior.xy
        ax.fill(x, y, color=color, alpha=alpha, edgecolor=edgecolor, linewidth=linewidth)

        for interior in geom.interiors:
            xi, yi = interior.xy
            ax.fill(xi, yi, color="white", alpha=1.0)

    elif geom.geom_type == "MultiPolygon":
        for poly in geom.geoms:
            _plot_polygon(
                ax,
                poly,
                color=color,
                alpha=alpha,
                edgecolor=edgecolor,
                linewidth=linewidth
            )

    elif geom.geom_type == "GeometryCollection":
        for g in geom.geoms:
            if g.geom_type in ["Polygon", "MultiPolygon"]:
                _plot_polygon(
                    ax,
                    g,
                    color=color,
                    alpha=alpha,
                    edgecolor=edgecolor,
                    linewidth=linewidth
                )


def plot_results(C1, C2, C3, disks, regions, results_table):
    """
    Plot:
    - the three disks
    - overlap regions
    - centroids
    - radial lines
    - edge points used for radial width
    """
    fig, ax = plt.subplots(figsize=(8, 8))

    disk_colors = ["red", "blue", "green"]

    for disk, color in zip(disks, disk_colors):
        _plot_polygon(
            ax,
            disk,
            color=color,
            alpha=0.15,
            edgecolor=color,
            linewidth=1.5
        )

    region_colors = {
        "Triple": "purple",
        "BlueGreenMinusRed": "cyan",
        "RedBlueMinusGreen": "orange",
    }

    for name, region in regions.items():
        _plot_polygon(
            ax,
            region,
            color=region_colors.get(name, "gray"),
            alpha=0.55,
            edgecolor="black",
            linewidth=1.0
        )

    # Convert table to iterable
    if pd is not None and isinstance(results_table, pd.DataFrame):
        rows = results_table.to_dict(orient="records")
    else:
        rows = results_table

    origin = np.array(C1, dtype=float)

    for row in rows:
        name = row["Region"]

        cx = row["Centroid_x"]
        cy = row["Centroid_y"]

        if np.isnan(cx) or np.isnan(cy):
            continue

        centroid = np.array([cx, cy], dtype=float)

        p1 = row["Edge_point_1"]
        p2 = row["Edge_point_2"]

        # Centroid
        ax.scatter(cx, cy, s=60, color="black", zorder=5)
        ax.text(cx, cy, f"  {name}", fontsize=9, va="center")

        # Line from C1 to centroid
        ax.plot(
            [origin[0], centroid[0]],
            [origin[1], centroid[1]],
            linestyle="--",
            linewidth=1.5,
            color="black"
        )

        # Edge-to-edge segment
        if p1 is not None and p2 is not None:
            if not np.any(np.isnan(p1)) and not np.any(np.isnan(p2)):
                ax.plot(
                    [p1[0], p2[0]],
                    [p1[1], p2[1]],
                    linewidth=2.5,
                    color="black"
                )

                ax.scatter(
                    [p1[0], p2[0]],
                    [p1[1], p2[1]],
                    s=50,
                    color="yellow",
                    edgecolor="black",
                    zorder=6
                )

    # Disk centers
    centers = [C1, C2, C3]
    for i, C in enumerate(centers, start=1):
        ax.scatter(C[0], C[1], s=80, marker="x", color="black", zorder=7)
        ax.text(C[0], C[1], f"  C{i}", fontsize=10, fontweight="bold")

    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title("Overlap regions, centroids, and radial edge-to-edge width")
    ax.grid(True, alpha=0.3)

    # No legend box
    # ax.legend(...) intentionally removed

    plt.show()


# ============================================================
# 7. Test example
# ============================================================

C1 = (0.0, 0.0)
C2 = (0.0, -0.009)
C3 = (0.021, -0.009)

R1 = 0.04
R2 = (0.02807 + 0.0398) / 2
R3 = 0.04

disks, regions, results_table = analyze_three_disks(
    C1, R1,
    C2, R2,
    C3, R3,
    resolution=2048,
    origin_index=1
)

print(results_table)

plot_results(C1, C2, C3, disks, regions, results_table)

In [ ]:
# ============================================================
# Mathematica expression translated to Python
# ============================================================

import numpy as np

# ------------------------------------------------------------
# Input values
# ------------------------------------------------------------

h = 0.00172

x1y1 = np.array([-0.021716267721258238, -0.007167915773105695], dtype=float)
x2y2 = np.array([0.006890967853044323, -0.008246070308953218], dtype=float)

rho01 = np.linalg.norm(x1y1)
theta01 = np.mod(np.arctan2(x1y1[1], x1y1[0]), 2.0 * np.pi)

rho02 = np.linalg.norm(x2y2)
theta02 = np.mod(np.arctan2(x2y2[1], x2y2[0]), 2.0 * np.pi)

B01 = 0.006
B02 = 0.006

A01 = 0.0008429971011798
A02 = 0.002656629827314824

Omega = 2.0 * np.pi * 60.0

R = 0.04
M = 0.0254          # Defined as in Mathematica, although it does not appear in the expression
sigma = 31.95973e6

I = 1j


# ------------------------------------------------------------
# Complex prefactor multiplying sin(phi)
# ------------------------------------------------------------

numerator = (
    I

    * np.exp(I * (theta01 + theta02))
    * (-np.exp(2.0 * I * theta01) + np.exp(2.0 * I * theta02))
    * rho01
    * (-R**2 + rho01**2)
    * rho02
    * (R**2 - rho02**2)
)

denominator = (
    4.0
    * np.pi
    * (
        -np.exp(I * theta02) * R**2
        + np.exp(I * theta01) * rho01 * rho02
    )
    * (
        np.exp(I * theta01) * R**2
        - np.exp(I * theta02) * rho01 * rho02
    )
    * (
        np.exp(2.0 * I * theta01) * rho01 * rho02
        + np.exp(2.0 * I * theta02) * rho01 * rho02
        - np.exp(I * (theta01 + theta02)) * (rho01**2 + rho02**2)
    )
)

complex_prefactor = numerator / denominator

# Remove tiny imaginary numerical noise if the result is essentially real
real_prefactor = np.real_if_close(complex_prefactor, tol=1000)


# ------------------------------------------------------------
# Final expression as a function of phi
# ------------------------------------------------------------

def mathematica_expression(phi):
    """
    Python version of the Mathematica expression.

    Parameters
    ----------
    phi : float
        Phase angle in radians.

    Returns
    -------
    value : float or complex
        Value of the expression.
    """
    return real_prefactor * np.sin(phi)


# ------------------------------------------------------------
# Print results
# ------------------------------------------------------------

print("rho01 =", rho01)
print("theta01 [rad] =", theta01)
print("theta01 [deg] =", np.degrees(theta01))

print("rho02 =", rho02)
print("theta02 [rad] =", theta02)
print("theta02 [deg] =", np.degrees(theta02))

print("\nComplex prefactor =", complex_prefactor)
print("Real prefactor    =", real_prefactor)

print("\nFinal expression:")
print("Expression(phi) = real_prefactor * sin(phi)")
print(f"Expression(phi) ≈ {real_prefactor} * sin(phi)")


# ------------------------------------------------------------
# Example evaluation
# ------------------------------------------------------------


In [ ]:
# ============================================================
# Numerical computation of the induced torque integrals
# Python version of Mathematica NIntegrate section
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad


# ============================================================
# 1. Parameters
# ============================================================
rho01 = np.linalg.norm(x1y1)
theta01 = np.mod(np.arctan2(x1y1[1], x1y1[0]), 2.0 * np.pi)

rho02 = np.linalg.norm(x2y2)
theta02 = np.mod(np.arctan2(x2y2[1], x2y2[0]), 2.0 * np.pi)

B01 = 0.006
B02 = 0.006

A01 = 0.0008429971011798
A02 = 0.002656629827314824


R11 = 0.023
R12 = 0.04
theta11 = ((R12**2 - R11**2)*theta01 - A01)/((R12**2 - R11**2))
theta12 = 2*theta01 - theta11

R21 = 0.001
R22 = 0.04
theta21 = ((R22**2 - R21**2)*theta02 - A02)/((R22**2 - R21**2))
theta22 = 2*theta02 - theta21

R = 0.04

sigma = 31.95973e6
Omega = 2.0 * np.pi * 60.0

# In your Mathematica code, lowercase omega appears as \[Omega] or \[Omega]-like symbolic factor.
# Assign a numerical value here if needed.
omega = 1.0

rho_eps = 0.0001
theta_eps = 0.00001


# ============================================================
# 2. Complex Arg and derivative of Arg
# ============================================================

def Arg(z):
    """
    Mathematica-like Arg for complex numbers.
    """
    return np.angle(z)


def dArg_dz(z):
    """
    Complex derivative convention corresponding to:

        Derivative[1][Arg][z]

    This is the complex derivative convention:

        d Arg(z) / dz = -i / (2 z)

    This is used to reproduce the Mathematica structure containing
    Derivative[1][Arg][...].
    """
    return -0.5j / z


# ============================================================
# 3. Source kernel
# ============================================================

def source_sp(rho, theta, Rin, Rout, theta1, theta2, R):
    """
    This function is the Python translation of sp11, sp22, sp12, sp21.

    The source sector is defined by:
        Rin <= rho <= Rout
        theta1 <= theta <= theta2

    In the Mathematica code, the final NIntegrate is of the form:

        NIntegrate[sp * rho, {rho, ...}, {theta, ...}]

    Therefore this function returns sp.
    The extra polar Jacobian rho is multiplied later in the integration.
    """

    a = Rin
    b = Rout

    d1 = theta - theta1
    d2 = theta - theta2

    s1 = np.sin(d1)
    s2 = np.sin(d2)

    c1 = np.cos(d1)
    c2 = np.cos(d2)

    s21 = np.sin(2.0 * d1)
    s22 = np.sin(2.0 * d2)

    c21 = np.cos(2.0 * d1)
    c22 = np.cos(2.0 * d2)

    # --------------------------------------------------------
    # Complex arguments
    # --------------------------------------------------------

    z_unit_1 = 1.0 - c1 + 1j * s1
    z_unit_2 = 1.0 - c2 + 1j * s2

    z_inner_a_1 = 1.0 - (a * c1) / rho + 1j * (a * s1) / rho
    z_inner_a_2 = 1.0 - (a * c2) / rho + 1j * (a * s2) / rho

    den_inner_a_1 = 1.0 + a**2 / rho**2 - (2.0 * a * c1) / rho
    den_inner_a_2 = 1.0 + a**2 / rho**2 - (2.0 * a * c2) / rho

    z_image_a_1 = 1.0 - (a * rho * c1) / R**2 + 1j * (a * rho * s1) / R**2
    z_image_a_2 = 1.0 - (a * rho * c2) / R**2 + 1j * (a * rho * s2) / R**2

    den_image_a_1 = 1.0 + (a**2 * rho**2) / R**4 - (2.0 * a * rho * c1) / R**2
    den_image_a_2 = 1.0 + (a**2 * rho**2) / R**4 - (2.0 * a * rho * c2) / R**2

    z_outer_b_1 = 1.0 - (rho * c1) / b + 1j * (rho * s1) / b
    z_outer_b_2 = 1.0 - (rho * c2) / b + 1j * (rho * s2) / b

    den_outer_b_1 = 1.0 + rho**2 / b**2 - (2.0 * rho * c1) / b
    den_outer_b_2 = 1.0 + rho**2 / b**2 - (2.0 * rho * c2) / b

    z_image_b_1 = 1.0 - (b * rho * c1) / R**2 + 1j * (b * rho * s1) / R**2
    z_image_b_2 = 1.0 - (b * rho * c2) / R**2 + 1j * (b * rho * s2) / R**2

    den_image_b_1 = 1.0 + (b**2 * rho**2) / R**4 - (2.0 * b * rho * c1) / R**2
    den_image_b_2 = 1.0 + (b**2 * rho**2) / R**4 - (2.0 * b * rho * c2) / R**2

    # --------------------------------------------------------
    # First main block
    # --------------------------------------------------------

    main_block = (
        (
            (-theta1 + theta2) * rho
            + (
                -b
                + a**3 / (3.0 * rho**2)
                + 8.0 * rho / 3.0
            )
        )
        * (s1 - s2)
        + (
            a**4 / (4.0 * rho**3)
            + 3.0 * rho / 4.0
            - rho * np.log(b / rho)
        )
        * (s21 - s22)
        - 8.0
        * rho
        * (
            (1.0 / 3.0) * s1
            - 0.5 * Arg(z_unit_1) * s1**2
            + (3.0 / 32.0) * s21
            - (1.0 / 3.0) * s2
            + 0.5 * Arg(z_unit_2) * s2**2
            - (3.0 / 32.0) * s22
        )
    )

    # --------------------------------------------------------
    # Inner radius contribution
    # --------------------------------------------------------

    inner_a_block = (
        -rho * Arg(z_inner_a_1) * c21 / a**2
        + rho * Arg(z_inner_a_2) * c22 / a**2
        - s1 / (2.0 * a)
        + a * s1 / (3.0 * rho**2)
        + a**2 * s21 / (4.0 * rho**3)
        - (
            rho**2
            * (
                -2.0 * a**2 / rho**3
                + 2.0 * a * c1 / rho**2
            )
            * s21
        )
        / (4.0 * a**2 * den_inner_a_1)
        - rho * np.log(den_inner_a_1) * s21 / (2.0 * a**2)
        + s2 / (2.0 * a)
        - a * s2 / (3.0 * rho**2)
        - a**2 * s22 / (4.0 * rho**3)
        + (
            rho**2
            * (
                -2.0 * a**2 / rho**3
                + 2.0 * a * c2 / rho**2
            )
            * s22
        )
        / (4.0 * a**2 * den_inner_a_2)
        + rho * np.log(den_inner_a_2) * s22 / (2.0 * a**2)
        + 0.5
        * (1.0 - rho**2 * c21 / a**2)
        * ((a * c1) / rho**2 - 1j * (a * s1) / rho**2)
        * dArg_dz(z_inner_a_1)
        - 0.5
        * (1.0 - rho**2 * c22 / a**2)
        * ((a * c2) / rho**2 - 1j * (a * s2) / rho**2)
        * dArg_dz(z_inner_a_2)
    )

    # --------------------------------------------------------
    # Image contribution associated with inner radius
    # --------------------------------------------------------

    image_a_block = (
        R**4 * Arg(z_image_a_1) * c21 / (a**2 * rho**3)
        - R**4 * Arg(z_image_a_2) * c22 / (a**2 * rho**3)
        + R**2 * s1 / (2.0 * a * rho**2)
        - (
            R**4
            * (
                2.0 * a**2 * rho / R**4
                - 2.0 * a * c1 / R**2
            )
            * s21
        )
        / (4.0 * a**2 * rho**2 * den_image_a_1)
        + R**4 * np.log(den_image_a_1) * s21 / (2.0 * a**2 * rho**3)
        - R**2 * s2 / (2.0 * a * rho**2)
        + (
            R**4
            * (
                2.0 * a**2 * rho / R**4
                - 2.0 * a * c2 / R**2
            )
            * s22
        )
        / (4.0 * a**2 * rho**2 * den_image_a_2)
        - R**4 * np.log(den_image_a_2) * s22 / (2.0 * a**2 * rho**3)
        + 0.5
        * (1.0 - R**4 * c21 / (a**2 * rho**2))
        * (-(a * c1) / R**2 + 1j * a * s1 / R**2)
        * dArg_dz(z_image_a_1)
        - 0.5
        * (1.0 - R**4 * c22 / (a**2 * rho**2))
        * (-(a * c2) / R**2 + 1j * a * s2 / R**2)
        * dArg_dz(z_image_a_2)
    )

    # --------------------------------------------------------
    # Outer radius contribution
    # --------------------------------------------------------

    outer_b_block = (
        rho * Arg(z_outer_b_1) * c21 / b**2
        - rho * Arg(z_outer_b_2) * c22 / b**2
        + s1 / (2.0 * b)
        + rho * s21 / (2.0 * b**2)
        - (
            rho**2
            * (
                2.0 * rho / b**2
                - 2.0 * c1 / b
            )
            * s21
        )
        / (4.0 * b**2 * den_outer_b_1)
        - rho * np.log(den_outer_b_1) * s21 / (2.0 * b**2)
        - s2 / (2.0 * b)
        - rho * s22 / (2.0 * b**2)
        + (
            rho**2
            * (
                2.0 * rho / b**2
                - 2.0 * c2 / b
            )
            * s22
        )
        / (4.0 * b**2 * den_outer_b_2)
        + rho * np.log(den_outer_b_2) * s22 / (2.0 * b**2)
        + 0.5
        * (-1.0 + rho**2 * c21 / b**2)
        * (-(c1) / b + 1j * s1 / b)
        * dArg_dz(z_outer_b_1)
        - 0.5
        * (-1.0 + rho**2 * c22 / b**2)
        * (-(c2) / b + 1j * s2 / b)
        * dArg_dz(z_outer_b_2)
    )

    # --------------------------------------------------------
    # Image contribution associated with outer radius
    # --------------------------------------------------------

    image_b_block = (
        R**4 * Arg(z_image_b_1) * c21 / (b**2 * rho**3)
        - R**4 * Arg(z_image_b_2) * c22 / (b**2 * rho**3)
        + R**2 * s1 / (2.0 * b * rho**2)
        - (
            R**4
            * (
                2.0 * b**2 * rho / R**4
                - 2.0 * b * c1 / R**2
            )
            * s21
        )
        / (4.0 * b**2 * rho**2 * den_image_b_1)
        + R**4 * np.log(den_image_b_1) * s21 / (2.0 * b**2 * rho**3)
        - R**2 * s2 / (2.0 * b * rho**2)
        + (
            R**4
            * (
                2.0 * b**2 * rho / R**4
                - 2.0 * b * c2 / R**2
            )
            * s22
        )
        / (4.0 * b**2 * rho**2 * den_image_b_2)
        - R**4 * np.log(den_image_b_2) * s22 / (2.0 * b**2 * rho**3)
        + 0.5
        * (1.0 - R**4 * c21 / (b**2 * rho**2))
        * (-(b * c1) / R**2 + 1j * b * s1 / R**2)
        * dArg_dz(z_image_b_1)
        - 0.5
        * (1.0 - R**4 * c22 / (b**2 * rho**2))
        * (-(b * c2) / R**2 + 1j * b * s2 / R**2)
        * dArg_dz(z_image_b_2)
    )

    # --------------------------------------------------------
    # Complete sp expression
    # --------------------------------------------------------

    F = (
        main_block
        - a**2 * inner_a_block
        - a**2 * image_a_block
        + b**2 * outer_b_block
        + b**2 * image_b_block
        - 2.0 * np.pi * rho
    )

    sp = rho * F

    return sp


# ============================================================
# 4. Complex numerical integration
# ============================================================

def integrate_complex_2d(
    integrand,
    rho_min,
    rho_max,
    theta_min,
    theta_max,
    epsabs=1e-11,
    epsrel=1e-7,
    limit=200
):
    """
    Numerically integrate a complex-valued function over:

        rho_min <= rho <= rho_max
        theta_min <= theta <= theta_max

    The integration is done by integrating real and imaginary parts separately.
    """

    def inner_real(rho):
        value, _ = quad(
            lambda theta: np.real(integrand(rho, theta)),
            theta_min,
            theta_max,
            epsabs=epsabs,
            epsrel=epsrel,
            limit=limit
        )
        return value

    def inner_imag(rho):
        value, _ = quad(
            lambda theta: np.imag(integrand(rho, theta)),
            theta_min,
            theta_max,
            epsabs=epsabs,
            epsrel=epsrel,
            limit=limit
        )
        return value

    real_part, real_err = quad(
        inner_real,
        rho_min,
        rho_max,
        epsabs=epsabs,
        epsrel=epsrel,
        limit=limit
    )

    imag_part, imag_err = quad(
        inner_imag,
        rho_min,
        rho_max,
        epsabs=epsabs,
        epsrel=epsrel,
        limit=limit
    )

    return real_part + 1j * imag_part, real_err, imag_err


# ============================================================
# 5. Coefficient calculation
# ============================================================

def compute_torque_coefficient(
    source_sector,
    target_sector,
    R,
    rho_eps=1e-4,
    theta_eps=1e-5,
    epsabs=1e-11,
    epsrel=1e-7
):
    """
    Computes the coefficient:

        c = 1/(2 pi) * Integral[sp_source(rho, theta) * rho dtheta drho]

    where:
    - source_sector defines the source angular/radial sector.
    - target_sector defines the region of integration.

    This reproduces the structure:

        pref * NIntegrate[sp * rho, ...]

    from Mathematica, where pref contains 1/(2 pi).
    """

    Rin_s, Rout_s, theta1_s, theta2_s = source_sector
    Rin_t, Rout_t, theta1_t, theta2_t = target_sector

    rho_min = Rin_t + rho_eps
    rho_max = Rout_t - rho_eps

    theta_min = theta1_t + theta_eps
    theta_max = theta2_t - theta_eps

    def integrand(rho, theta):
        return source_sp(
            rho=rho,
            theta=theta,
            Rin=Rin_s,
            Rout=Rout_s,
            theta1=theta1_s,
            theta2=theta2_s,
            R=R
        ) * rho

    integral, real_err, imag_err = integrate_complex_2d(
        integrand,
        rho_min,
        rho_max,
        theta_min,
        theta_max,
        epsabs=epsabs,
        epsrel=epsrel
    )

    coefficient = integral / (2.0 * np.pi)

    return coefficient, integral, real_err, imag_err


# ============================================================
# 6. Define source and target sectors
# ============================================================

sector_11 = (R11, R12, theta11, theta12)
sector_22 = (R21, R22, theta21, theta22)


# ============================================================
# 7. Compute T11, T22, T12, T21 coefficients numerically
# ============================================================

# T11: source 11 integrated over sector 11
c11, I11, err11_re, err11_im = compute_torque_coefficient(
    source_sector=sector_11,
    target_sector=sector_11,
    R=R,
    rho_eps=rho_eps,
    theta_eps=theta_eps
)

# T22: source 22 integrated over sector 22
c22, I22, err22_re, err22_im = compute_torque_coefficient(
    source_sector=sector_22,
    target_sector=sector_22,
    R=R,
    rho_eps=rho_eps,
    theta_eps=theta_eps
)

# T12: source 11 integrated over sector 22
c12, I12, err12_re, err12_im = compute_torque_coefficient(
    source_sector=sector_11,
    target_sector=sector_22,
    R=R,
    rho_eps=rho_eps,
    theta_eps=theta_eps
)

# T21: source 22 integrated over sector 11
c21, I21, err21_re, err21_im = compute_torque_coefficient(
    source_sector=sector_22,
    target_sector=sector_11,
    R=R,
    rho_eps=rho_eps,
    theta_eps=theta_eps
)


print("Numerical coefficients:")
print("c11 =", c11)
print("c22 =", c22)
print("c12 =", c12)
print("c21 =", c21)

print("\nIntegration estimates:")
print("I11 =", I11, "errors:", err11_re, err11_im)
print("I22 =", I22, "errors:", err22_re, err22_im)
print("I12 =", I12, "errors:", err12_re, err12_im)
print("I21 =", I21, "errors:", err21_re, err21_im)


# ============================================================
# 8. Torque terms as functions of time and phase
# ============================================================

def T11(t):
    return (
        c11
        * B01**2
        * sigma
        * omega
        * np.cos(t * Omega)**2
    )


def T22(t, phi):
    return (
        c22
        * B02**2
        * sigma
        * omega
        * np.cos(phi + t * Omega)**2
    )


def T12(t, phi):
    return (
        c12
        * B01
        * B02
        * sigma
        * omega
        * np.cos(t * Omega)
        * np.cos(phi + t * Omega)
    )


def T21(t, phi):
    return (
        c21
        * B01
        * B02
        * sigma
        * omega
        * np.cos(t * Omega)
        * np.cos(phi + t * Omega)
    )


def total_torque_complex(t, phi):
    return T11(t) + T22(t, phi) + T12(t, phi) + T21(t, phi)


def total_torque_real(t, phi):
    return np.real(total_torque_complex(t, phi))


# ============================================================
# 9. Example evaluation
# ============================================================

t_example = 0.0
phi_example = np.pi / 2.0

print("\nExample evaluation:")
print("T11 =", T11(t_example))
print("T22 =", T22(t_example, phi_example))
print("T12 =", T12(t_example, phi_example))
print("T21 =", T21(t_example, phi_example))
print("Total torque complex =", total_torque_complex(t_example, phi_example))
print("Total torque real    =", total_torque_real(t_example, phi_example))


# ============================================================
# 10. Optional plot
# ============================================================

t_values = np.linspace(0.0, 1.0 / 60.0, 1000)
phi_value = np.pi / 2.0

torque_values = total_torque_real(t_values, phi_value)

plt.figure(figsize=(8, 4))
plt.plot(t_values, torque_values)
plt.xlabel("Time t [s]")
plt.ylabel("Total torque")
plt.title("Total induced torque computed from numerical integrals")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
phi=0.07
velmax=-real_prefactor*A01*A02*Omega*np.sin(phi)/((c11+(c12+c21)*np.cos(phi)+c22))

print(velmax)


In [ ]:
# ============================================================
# Plot velmax as a function of phi
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Phi range
# ------------------------------------------------------------

phi_values = np.linspace(0.001, 1/4*np.pi+0.5, 2000)

# ------------------------------------------------------------
# Function velmax(phi)
# ------------------------------------------------------------

def velmax_function(phi):
    denominator = c11 + (c12 + c21) * np.cos(phi) + c22

    velmax = (
        -real_prefactor * A01 * A02 * Omega * np.sin(phi)
        / denominator
    )

    # If the result has tiny imaginary numerical noise, keep only real part
    return np.real(velmax)


velmax_values = velmax_function(phi_values)

# ------------------------------------------------------------
# Evaluate your specific value phi = 0.07
# ------------------------------------------------------------

phi_point = 0.07
velmax_point = velmax_function(phi_point)

print("velmax(phi = 0.07) =", velmax_point)

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

plt.figure(figsize=(8, 5))

plt.plot(phi_values, velmax_values, linewidth=2, label=r"$\omega_{\max}(\phi)$")

# Mark phi = 0.07
plt.scatter(phi_point, velmax_point, color="black", zorder=5)
plt.axvline(phi_point, linestyle="--", color="black", alpha=0.6)

plt.xlabel(r"$\phi$ [rad]")
plt.ylabel(r"$\omega_{\max}$")
plt.title(r"Maximum angular velocity as a function of $\phi$")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()